# XNAT-Jupyter-CS demo workflow

Important: Launch Jupyter notebook from XNAT project level to run this notebook.

This notebook demonstrates how to:
* analyze an XNAT project for structural scans and segmentations, 
* build a list of scans to process with custom user analyses,
* develop a custom user meta-script to run for each of the scans/sessions,
* launch containerized batch processing for all scans.

## 1. Mandatory user-settable variables
Edit this cell to set all mandatory notebook variables here. 

In [8]:
#set to True to regenerate project directory structure saved in local json file.
rebuild_directory_structure=True

#XNAT project label
project='NSCLCRadiomicsDemo'

#Persistent workspace root path
root_dir=Path("/workspace/mmilchenko")

## 2. General initializations
Just run this cell for notebook environment set up.

In [9]:
import os, subprocess, sys
from pathlib import Path

#Library with XNAT Jupyter workflow Python and shell scripts.
pymipl_path = os.path.abspath('../')
sys.path.append(pymipl_path)
sys.path.append( os.path.abspath(pymipl_path+'/xnat_workflow') )

#dicom_sort is part of pymipl. dicom_sort can automatically analyze XNAT projects for structural scans and segmentations.
from dicom_sort import *

#Derived variable initializations
local_workdir_path=root_dir / project
xnat_project_path=f'/data/projects/{project}/experiments'
directory_structure_file=local_workdir_path / "project_dir_structure.json"
xnat_structure_file=local_workdir_path / "xnat_structure.json"
scanlist_file=local_workdir_path / "scans.csv"



## 3. Create a list of structural scans and associated segmentations.
Run the next cell to Project structure, with all DICOM scans and segmentations, is written to  configuration files in workspace project location.


In [11]:
# analyze_dir finds all structural scans and segmentations (DICOM RTSTRUCT and DICOM Segmentation Object) in the project. 
# The results are saved into a JSON file for quick rerun. 
if rebuild_directory_structure:
    os.makedirs(os.path.dirname(directory_structure_file), exist_ok=True)    
    d=analyze_dir(xnat_project_path,directory_structure_file)
else:
    #load analyzed directory structure from disk
    with open(directory_structure_file, 'r') as file:
        d = json.load(file)

#This writes out human-readable list of structural scans with segmentations into a csv file.
subjects,scans=reindex_to_structurals_and_segs(d,xnat_structure_file,scanlist_file)
print (f'Number of structural scans: {len(scans)}')
print ('First scan: ',scans[0])

/data/projects/NSCLCRadiomicsDemo/experiments/09-18-2008-StudyID-NA-69331
/data/projects/NSCLCRadiomicsDemo/experiments/01-01-2014-StudyID-NA-85095
/data/projects/NSCLCRadiomicsDemo/experiments/01-01-2014-StudyID-NA-34270
Number of structural scans: 3
First scan:  {'Subject': 'LUNG1-001', 'Experiment': '09-18-2008-StudyID-NA-69331', 'StructScan': '0', 'StructScanSerDesc': '', 'SegScan1': '3', 'SegScan1_SerDesc': '', 'SegScan1_SOPClass': 'RTStruct', 'SegScan2': '300', 'SegScan2_SerDesc': 'Segmentation', 'SegScan2_SOPClass': 'Seg'}


## 4. Workflow inititalizations

In [7]:
global_vars={}
#env_type='jupyter'
env_type='container'
project='NSCLC_RADIOMICS'
workflow_id='nsclc-segmentation-codebase-20260317'

xnat_command_id=20
xnat_command_wrapper_id=26 #'xnat-ai-workflow' #26

global_vars['g_workflow_id']=workflow_id
root_dir=Path('/workspace/mmilchenko')
local_workdir_path=Path('/workspace/mmilchenko') / project

#user micromamba environment
user_env_repo='gevaert'
#user (re)source directory
user_src_repo='nsclc-segmentation-codebase-20260317'

global_vars['g_user_env_repo']=user_env_repo
global_vars['g_user_src_repo']=user_src_repo

if env_type=='jupyter':

    #path that mounts directory with XNAT experiments
    global_vars['g_input_mount_path']=Path('/data/projects') / project / 'experiments'
    #path to local directory where processing will be stored
    global_vars['g_local_workdir_path']=Path('/workspace/mmilchenko') / project
    #library locations, algorithm specific
    global_vars['g_pymipl_dir']=root_dir / "pymipl"
    #main algorithm repository dir
    global_vars['g_env_repo_dir']=root_dir / 'envs' / user_env_repo
    global_vars['g_alg_repo_dir']=root_dir / 'src' / user_src_repo
    global_vars['g_project']=project
        
elif env_type == 'container': #built-in defaults used in the bootstrap image.
    wa.init_global_vars_bootstrap_image(global_vars,project)
    


### 4. Create step descriptors for the XNAT workflow parser.
This creates batch files to run on experiment specific containers. Set flags in the beginning to control execution.

Performs the following:
1. convert strcutrual DICOM's to NIFTI
2. convert existing segmentations, if any, to NIFTI
3. run AI segmentation
4. generate QC images
5. compute Dice coefficients.

The batch file is then run outside of this notebook by external containers.

In [17]:
import importlib
import workflow_adapters as wa
importlib.reload(wa)

set_logger()
dt=datetime.datetime.now().strftime("%Y%m%d_%H%M")
batch_file=local_workdir_path / f"batch_{dt}.sh"
#n indicates starting position in the spreadsheet.
n=0

xnat_interface=None
start_pos=170
end_pos=9999
subjects={}
nExp=0
nFailed=0
sessions_failed=[]
num_sessions=len(scans)

for scan in scans:
    n=n+1
    if n<start_pos or n>end_pos:  continue
    job,steps={},[]
    
    #Job-level variables, with 'job_' prefix
    #TODO. develop necessary abstractions, make this job assignment block into a function and move it to workflow_adapters.py
    
    job_experiment=scan['Experiment']

    #Do not change the next three lines to correctly preserve the scan context
    job_scan_context=global_vars['g_input_mount_path'] #/ job_scan_id / 'DICOM'
    if env_type == 'jupyter': job_scan_context = job_scan_context / Path(job_experiment)
    job_scan_context = job_scan_context / Path('SCANS')    
    
    job['job_title']=f"Workflow {workflow_id}, subject {scan['Subject']}, experiment {scan['Experiment']}"
    job['job_struct_path'] = job_scan_context / scan['StructScan'] / 'DICOM'
    #job['job_seg1_path'] = job_scan_context / scan['SegScan1'] / 'DICOM'
    #job['job_seg2_path'] = job_scan_context / scan['SegScan2'] / 'DICOM'
    job['job_subject'] = scan['Subject']
    job['job_exp_label'] = scan['Experiment']
    job_id=f"{workflow_id}_{scan['Subject']}_{scan['Experiment']}"
    job['job_id']=job_id
    job['job_workdir']=global_vars['g_local_workdir_path'] / scan['Subject'] / scan['Experiment']

    #Step 0. Test mounted environment
    #step={"step_title": "1. ls /opt/packages/user/env_repo"}
    #step['step_command']="ls /opt/packages/user/env_repo"
    #steps+=[step]

    #Step 0-1. Test mounted environment
    #step={"step_title": "1. ls /opt/packages/user/alg_repo"}
    #step['step_command']="ls /opt/packages/user/alg_repo"
    #steps+=[step]    

    #TODO. Make step generation into a function, as some parts are clearly automatable.
    #Step 1. Run segmentation.    
    step={"step_title": "1. Run NSCLC tumor segmentation on structural scan."}
    step['step_command']="micromamba run -p {g_env_repo_dir} python {g_alg_repo_dir}/run_segmentation.py \
        --input {job_struct_path} --output-dir {job_workdir} --output-format nrrd --verbose"
    steps+=[step]

    #Step 2. Convert structural to nifti
    step={"step_title": "Convert structural to NIFTI"}
    step['step_command']="micromamba run -n base python {g_pymipl_dir}/test_rt-utils.py {job_struct_path} {job_workdir}/ct"
    steps+=[ step ]

    step={"step_title": "adjust lesion mask location"}
    step['step_command']="mv {job_workdir}/DICOM/lesion_mask.nii.gz {job_workdir}/lesion_mask.nii.gz"
    steps+=[ step ]
    
    #Step 3. Generate QC image.
    step={"step_title": "2. Generate QC image."}
    step['step_command']="micromamba run -n base python {g_pymipl_dir}/slice_qc.py  \
        -o {job_workdir}/qc.png --mask {job_workdir}/lesion_mask.nii.gz {job_workdir}/ct_struct.nii" 
    steps+=[step]

    #Step 4. Clean up.
    step={"step_title": "3. Clean up"}
    step['step_command']="rm -r {job_workdir}/ct_struct.nii {job_workdir}/DICOM"
    steps+=[step]

    #Step 5. Upload the results back to XNAT.
    step={"step_title": "4. Upload results to XNAT"}
    step['step_command']='micromamba run -n base python {g_pymipl_dir}/xnat_workflow/sync-resource-with-xnat.py \
            --level experiment --project {g_project} --subject {job_subject} --experiment {job_exp_label} --local_resource {job_workdir} \
            --remote_resource {g_workflow_id} --create_hierarchy 1'
    steps+=[step]

    #process job and write out
    #TODO. this calls for another function to finalize job creation/file generation.
    job['steps']=steps
    job_id=f"{workflow_id}_{scan['Subject']}_{job_experiment}"
    local_job_dir=local_workdir_path / 'jobs' / job_id
    job_file_yaml=local_job_dir / 'job.yaml'
    job_file_sh=local_job_dir / 'job.sh'
    local_job_dir.mkdir(parents=True,exist_ok=True)
    
    with open(job_file_yaml,"w") as f:
        yaml.safe_dump(paths_to_str(job),f,sort_keys=False)

    #this also needs to be a dedicated function. 
    if env_type=='jupyter': #write all commands to a single batch file
        wa.workflow_to_batch(job,global_vars,batch_file)
        print(batch_file)
        break #DEBUG
        
    else: #one batch file per job
        #create batch        
        print(job_file_sh)
        #reset job script
        ! truncate -s 0 {job_file_sh}
        #generate job script
        #DEBUG
        wa.workflow_to_batch(job,global_vars,job_file_sh)
        ! chmod +x {job_file_sh}
        #store batch file.
        print('sending batch to xnat resource')
        #break #DEBUG
        #res=0
        #TODO this call is indicative of the mixture of scopes issue described above.
        if xnat_interface is None: xnat_interface=get_xnat_interface(project)
        #DEBUG: uncomment the next line later.
        res1=resource_to_xnat(local_job_dir, workflow_id, project, job['job_subject'], job['job_exp_label'],xnat_interface)
        print('submitting job to Container Service')
        #print(f"launching CS command for {project}, {job['job_subject']} ,{job['job_exp_label']}, {workflow_id}, {xnat_command_wrapper_id}") 
        #res2=launch_cs_command(xnat_interface,project,job['job_subject'],job['job_exp_label'],scan['session'],workflow_id,15,"xnat-ai-workflow",verbose=True)
        #res2=launch_cs_command(xnat_interface,project,job['job_subject'],job['job_exp_label'],scan['session'],workflow_id,15,26,verbose=True)
        exp_id=xnat_interface.select.project(project).subject(job['job_subject']).experiment(job['job_exp_label']).id()
        res2=1
        #res2=launch_cs_command(xnat_interface,project,job['job_subject'],job['job_exp_label'],exp_id,workflow_id,xnat_command_id,xnat_command_wrapper_id,verbose=True)
        #res=launch_cs_command(xnat_interface,"BM_WU","M00101754","M00101754_20180719144757","TAP02_E01992",workflow_id,15,"xnat-ai-workflow")     
        
        if res1 == 0 and res2:
            if n % 10 == 0: print(f'Done {n} out of {num_sessions} ({n*100/num_sessions:.1f}%)')
        else: 
            print (f'Failed sending configuration to session resource, error status 1 {res1}, error status 2 {res2}. Stopping execution.')        
    #break #DEBUG

/workspace/mmilchenko/NSCLC_RADIOMICS/jobs/nsclc-segmentation-codebase-20260317_LUNG1-275_09-18-2008-StudyID-NA-06365/job.sh
sending batch to xnat resource
submitting job to Container Service
Done 170 out of 422 (40.3%)
/workspace/mmilchenko/NSCLC_RADIOMICS/jobs/nsclc-segmentation-codebase-20260317_LUNG1-200_09-02-2007-NA-NA-56233/job.sh
sending batch to xnat resource
submitting job to Container Service
/workspace/mmilchenko/NSCLC_RADIOMICS/jobs/nsclc-segmentation-codebase-20260317_LUNG1-102_03-02-2006-StudyID-NA-37333/job.sh
sending batch to xnat resource
submitting job to Container Service
/workspace/mmilchenko/NSCLC_RADIOMICS/jobs/nsclc-segmentation-codebase-20260317_LUNG1-326_03-21-2009-StudyID-NA-81321/job.sh
sending batch to xnat resource
submitting job to Container Service
/workspace/mmilchenko/NSCLC_RADIOMICS/jobs/nsclc-segmentation-codebase-20260317_LUNG1-340_05-02-2009-StudyID-NA-86385/job.sh
sending batch to xnat resource
submitting job to Container Service
/workspace/mmilch

In [9]:
!pip install nibabel==5.3.0
#interface=get_xnat_interface(project)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 55.9 MB/s eta 0:00:00a 0:00:01


In [13]:
exp=xnat_interface.select.project(project).subject('LUNG1-093').experiment('04-13-2006-StudyID-NA-25111').id()

2026-03-26 20:01:32,219 - urllib3.connectionpool - DEBUG - https://tap.embarklabs.ai:443 "GET /data/projects/NSCLC_RADIOMICS/subjects/LUNG1-093/experiments?format=csv&columns=ID,label HTTP/1.1" 200 None
2026-03-26 20:01:32,219 - urllib3.connectionpool - DEBUG - https://tap.embarklabs.ai:443 "GET /data/projects/NSCLC_RADIOMICS/subjects/LUNG1-093/experiments?format=csv&columns=ID,label HTTP/1.1" 200 None


In [ ]:
pip install --force-reinstall --no-cache-dir "numpy<2" pandas matplotlib

In [29]:
!pip cache purge

Files removed: 6
